# MedSigLIP Fine-tuning on CheXpert + CQ500 (LoRA)
### Changes from v1:
1. `torch.amp.autocast("cuda")` replacing deprecated `torch.cuda.amp.autocast()`  
2. **LoRA fine-tuning** (PEFT) applied to the vision encoder only — rationale in dedicated cell  
3. **CQ500 head-CT support** — DICOM reading, HU windowing → RGB, slice extraction, report generation from `reads.csv`  
4. **Combined dataset** mixing CheXpert and CQ500  
5. **HuggingFace Hub upload** after final validation  
6. **Text-to-image retrieval** discussion and architectural choice to preserve it

In [ ]:
# Install all dependencies
!pip install -q kaggle transformers accelerate peft huggingface_hub pydicom nibabel

In [ ]:
import os, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, ConcatDataset
from transformers import AutoProcessor, AutoModel
from peft import LoraConfig, get_peft_model, TaskType
from PIL import Image
import pydicom
from pathlib import Path
from huggingface_hub import login, HfApi

In [ ]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using device: MPS (Apple)")
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Using device: CUDA (GPU) - {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print("Using device: CPU")

### HuggingFace Login & Model Load
MedSigLIP is gated. Accept the [Health AI Developer Foundations terms](https://huggingface.co/google/medsiglip-448) 
and set `HF_TOKEN` in Colab Secrets (**Runtime → Secrets**).

In [ ]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
login(token=HF_TOKEN)

MODEL_NAME = "google/medsiglip-448"

processor = AutoProcessor.from_pretrained(MODEL_NAME, token=HF_TOKEN)
model     = AutoModel.from_pretrained(MODEL_NAME, token=HF_TOKEN).to(device)

print(f"Vision encoder params: {sum(p.numel() for p in model.vision_model.parameters())/1e6:.0f}M")
print(f"Text encoder params:   {sum(p.numel() for p in model.text_model.parameters())/1e6:.0f}M")

## Design Decision: LoRA vs Full Fine-tuning

**Recommendation: LoRA applied to the vision encoder only. Text encoder frozen.**

### Why not full fine-tuning?

| Factor | Impact |
|---|---|
| MedSigLIP already pretrained on CXR + CT data | Marginal gains from full finetune; high risk of destroying pretrained features |
| Combined dataset is small (~223K CheXpert + 491 CQ500 scans) | ~2 orders of magnitude smaller than pretraining corpus — full finetune will overfit |
| 800M total parameters | Full finetune requires ~12 GB of optimizer state alone at fp32 |
| Text-to-image retrieval preservation | Moving the text encoder's embedding space will break cross-modal alignment |

### Why LoRA on vision only?

- Adds trainable rank-r matrices inside each attention block of the vision transformer;  
  all original weights stay **frozen** — no catastrophic forgetting.  
- The vision encoder needs to learn fine-grained CheXpert/CQ500 visual features;  
  the text encoder already represents radiology report language well.  
- Freezing text = **text-to-image retrieval is preserved by construction** (see cell below).  
- Typically adds only ~1–2% extra parameters (e.g. r=16 adds ~6M params to a 400M encoder).  

### Will text-to-image retrieval survive fine-tuning?

Yes, if the **text encoder is frozen** and training uses the **symmetric SigLIP loss**.  
The text embedding space stays identical to pretraining, so any existing text query will  
still produce a valid embedding. The vision encoder learns to move its image embeddings  
**closer to the correct text embeddings** — which is exactly what improves both  
image-to-text AND text-to-image retrieval simultaneously.  

If you were to unfreeze the text encoder, retrieval from arbitrary text queries  
(e.g. "pneumothorax") would degrade because the text embedding space would drift  
toward your narrow CheXpert/CQ500 vocabulary.

In [ ]:
# ─── Freeze everything first ─────────────────────────────────────────────
for p in model.parameters():
    p.requires_grad_(False)

# ─── Apply LoRA to vision encoder attention layers only ──────────────────
# target_modules: the attention projection names in SigLIP's ViT blocks.
# For google/medsiglip-448 these are the standard ViT attention projections.
lora_config = LoraConfig(
    r=16,                        # rank — 8 is more conservative, 32 is more expressive
    lora_alpha=32,               # scaling: alpha/r = 2 (standard choice)
    lora_dropout=0.05,
    # SigLIP ViT attention weights — query, key, value, output projections
    target_modules=["q_proj", "k_proj", "v_proj", "out_proj"],
    # We only want to adapt the vision_model sub-module
    # PEFT wraps the full model; we constrain via modules_to_save below
    bias="none",
)

# Wrap only the vision_model with LoRA
model.vision_model = get_peft_model(model.vision_model, lora_config)

# logit_scale and logit_bias are small scalars — fine to train directly
model.logit_scale.requires_grad_(True)
model.logit_bias.requires_grad_(True)

# Check trainable parameter count
total   = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

### Data Download

In [ ]:
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

import kaggle

# CheXpert
kaggle.api.dataset_download_files("ashery/chexpert", path="./CheXpert-v1.0-small", unzip=True)

# CQ500 — hosted by qure.ai, downloadable via their script or Kaggle mirror
# The official download is at http://headctstudy.qure.ai/dataset (requires Google sign-in)
# A Kaggle mirror: felipekitamura/head-ct-hemorrhage (note: subset; use official for full data)
# Adjust the path below to wherever you place the CQ500 folder.
CQ500_ROOT = "./CQ500"        # parent folder containing CQ500-CT-0/, CQ500-CT-1/, ...
CQ500_READS_CSV = "./CQ500/reads.csv"

print("Download paths set. Adjust CQ500_ROOT if needed.")

### CheXpert Data Loading

In [ ]:
train_full = pd.read_csv("/content/CheXpert-v1.0-small/train.csv").fillna(0)
val_full   = pd.read_csv("/content/CheXpert-v1.0-small/valid.csv").fillna(0)

print(f"CheXpert — Train: {len(train_full):,} | Val: {len(val_full):,}")

In [ ]:
def generate_chexpert_report(row):
    """Unchanged from v1 — converts CheXpert label row to radiology text."""
    labels  = row.iloc[5:]
    pos     = list(labels[labels ==  1.0].index)
    unc     = list(labels[labels == -1.0].index)

    age  = int(row["Age"])              if pd.notna(row["Age"])              else None
    sex  = row["Sex"].lower()           if pd.notna(row["Sex"])              else None
    view = row["Frontal/Lateral"].lower() if pd.notna(row["Frontal/Lateral"]) else None

    parts = [f"{view.capitalize()} chest radiograph" if view else "Chest radiograph"]
    demo  = []
    if age: demo.append(f"{age}-year-old")
    if sex: demo.append(sex)
    if demo: parts.append(f"of {' '.join(demo)} patient")

    if not pos and not unc:
        parts.append("demonstrates no acute cardiopulmonary abnormality")
    else:
        finds = []
        if pos: finds.append("shows " + ", ".join(f.lower().replace("_"," ") for f in pos))
        if unc: finds.append("possible " + ", ".join(f.lower().replace("_"," ") for f in unc))
        parts.append(". ".join(finds))

    return " ".join(parts) + "." 

### CQ500 Head CT Support

**Key differences from CheXpert:**

| Dimension | CheXpert | CQ500 |
|---|---|---|
| Modality | Chest X-ray (2D) | Head CT (3D DICOM volume) |
| File format | JPEG | DICOM (.dcm) |
| Preprocessing | Resize + normalize | HU windowing → 3-channel RGB + slice extraction |
| Labels | 14 binary finding columns | `reads.csv` with 3-reader consensus per 13 findings |
| Body region | Thorax | Brain |

**Windowing:** CT scanners output Hounsfield Units (HU). We apply three standard  
neuoradiology windows and stack them as RGB channels — the same preprocessing used  
in the original qure.ai paper and standard in head-CT deep learning:
- Brain window  (C=40, W=80)  → R channel  
- Subdural window (C=75, W=215) → G channel  
- Bone window (C=600, W=2800) → B channel  

**Slice selection:** each CQ500 study contains 30–100+ axial slices.  
We extract the **middle third** (most diagnostically informative for most findings)  
and pick the median slice as the representative 2D image.  
For production use, replace with a pathology-guided slice selector.

In [ ]:
# ────────────────────────────────────────────────────
# CQ500 label parsing
# reads.csv columns (after 'name'):
#   R1:ICH, R2:ICH, R3:ICH,
#   R1:IPH, R2:IPH, R3:IPH,
#   R1:IVH, R2:IVH, R3:IVH,
#   R1:SDH, R2:SDH, R3:SDH,
#   R1:EDH, R2:EDH, R3:EDH,
#   R1:SAH, R2:SAH, R3:SAH,
#   R1:BleedLocation-Left, R2:BleedLocation-Left, R3:BleedLocation-Left,
#   R1:BleedLocation-Right, ...
#   R1:CalvarialFracture, ..., R1:OtherFracture, ...,
#   R1:MassEffect, ..., R1:MidlineShift, ..., R1:ChronicBleed, ...
# ────────────────────────────────────────────────────

# Full label list (short names used as keys)
CQ500_FINDINGS = [
    "ICH", "IPH", "IVH", "SDH", "EDH", "SAH",
    "CalvarialFracture", "OtherFracture",
    "MassEffect", "MidlineShift", "ChronicBleed",
    "BleedLocation-Left", "BleedLocation-Right",
]

# Human-readable descriptions for report text
CQ500_FINDING_NAMES = {
    "ICH":                "intracranial hemorrhage",
    "IPH":                "intraparenchymal hemorrhage",
    "IVH":                "intraventricular hemorrhage",
    "SDH":                "subdural hematoma",
    "EDH":                "extradural hematoma",
    "SAH":                "subarachnoid hemorrhage",
    "CalvarialFracture":  "calvarial fracture",
    "OtherFracture":      "skull fracture",
    "MassEffect":         "mass effect",
    "MidlineShift":       "midline shift",
    "ChronicBleed":       "chronic bleed",
    "BleedLocation-Left": "left hemisphere bleed",
    "BleedLocation-Right":"right hemisphere bleed",
}


def parse_cq500_reads(reads_csv_path):
    """
    Parse reads.csv, compute majority vote consensus across 3 radiologists,
    and return a DataFrame with columns: [name, ICH, IPH, IVH, ...].
    Majority vote: a finding is positive if >= 2 of 3 readers marked it 1.
    """
    df = pd.read_csv(reads_csv_path)

    consensus = pd.DataFrame()
    consensus["name"] = df["name"]

    for finding in CQ500_FINDINGS:
        r_cols = [c for c in df.columns if finding in c]   # R1:ICH, R2:ICH, R3:ICH
        if len(r_cols) >= 2:
            # Majority vote: positive if sum of reads >= ceil(n_readers/2)
            consensus[finding] = (df[r_cols].fillna(0).sum(axis=1) >= 2).astype(int)
        elif len(r_cols) == 1:
            consensus[finding] = df[r_cols[0]].fillna(0).astype(int)
        else:
            consensus[finding] = 0

    return consensus


def generate_cq500_report(row):
    """
    Convert a CQ500 consensus label row into a natural language radiology text.
    Format mirrors generate_chexpert_report for stylistic consistency.
    """
    pos_findings = [CQ500_FINDING_NAMES[f] for f in CQ500_FINDINGS
                    if row.get(f, 0) == 1 and f in CQ500_FINDING_NAMES]

    if not pos_findings:
        return "Non-contrast head CT scan demonstrates no acute intracranial abnormality."

    finding_str = ", ".join(pos_findings)
    return f"Non-contrast head CT scan demonstrates {finding_str}." 

In [ ]:
def window_ct_slice(pixel_array, center, width):
    """Apply a single HU window to a 2D CT slice, return uint8 0-255."""
    lo = center - width / 2
    hi = center + width / 2
    windowed = np.clip(pixel_array.astype(np.float32), lo, hi)
    windowed = ((windowed - lo) / (hi - lo) * 255).astype(np.uint8)
    return windowed


def dicom_to_rgb_pil(dcm_path):
    """
    Read a DICOM slice and return a PIL RGB image using the 3-window stack:
      R = brain window    (C=40,  W=80)
      G = subdural window (C=75,  W=215)
      B = bone window     (C=600, W=2800)
    This is the standard preprocessing for head-CT deep learning.
    """
    dcm = pydicom.dcmread(str(dcm_path))
    raw = dcm.pixel_array.astype(np.float32)

    # Apply RescaleSlope / RescaleIntercept to get true HU values
    slope     = float(getattr(dcm, "RescaleSlope",     1))
    intercept = float(getattr(dcm, "RescaleIntercept", 0))
    hu = raw * slope + intercept

    r = window_ct_slice(hu, center=40,  width=80)     # brain
    g = window_ct_slice(hu, center=75,  width=215)    # subdural
    b = window_ct_slice(hu, center=600, width=2800)   # bone

    rgb = np.stack([r, g, b], axis=-1)
    return Image.fromarray(rgb, mode="RGB")


def get_representative_slice(scan_folder):
    """
    Given a folder of DICOM files for one study, return the PIL RGB image
    of the representative slice (median of the middle-third of sorted slices).
    Falls back to the middle slice if InstanceNumber is unavailable.
    """
    dcm_files = sorted(Path(scan_folder).rglob("*.dcm"))
    if not dcm_files:
        return None

    # Sort by InstanceNumber (axial position)
    def get_instance(f):
        try:
            return int(pydicom.dcmread(str(f), stop_before_pixels=True).InstanceNumber)
        except Exception:
            return 0

    dcm_files = sorted(dcm_files, key=get_instance)

    n = len(dcm_files)
    # Take the median slice of the middle third — most informative for most findings
    start = n // 3
    end   = 2 * n // 3
    mid_idx = (start + end) // 2
    chosen = dcm_files[mid_idx]

    return dicom_to_rgb_pil(chosen)

In [ ]:
class CQ500Dataset(torch.utils.data.Dataset):
    """
    Dataset for CQ500 head CT scans.

    Expected folder structure:
        CQ500_ROOT/
          CQ500-CT-0/           <- scan folder (may contain sub-folders with DICOMs)
          CQ500-CT-1/
          ...
          reads.csv             <- label file with 3-reader consensus columns

    Each item returns the same keys as MedSigLIPDataset for drop-in compatibility
    with the shared training loop.
    """

    def __init__(self, cq500_root, reads_csv_path):
        self.root = Path(cq500_root)
        self.labels_df = parse_cq500_reads(reads_csv_path).reset_index(drop=True)

        # Build (scan_folder, report) pairs — skip missing folders
        self.samples = []
        for _, row in self.labels_df.iterrows():
            scan_dir = self.root / row["name"]
            if scan_dir.exists():
                report = generate_cq500_report(row)
                self.samples.append((scan_dir, report))
            else:
                print(f"[CQ500] Folder not found, skipping: {scan_dir}")

        print(f"CQ500: {len(self.samples)} usable scans out of {len(self.labels_df)}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        scan_dir, report = self.samples[idx]

        image = get_representative_slice(scan_dir)
        if image is None:
            # Fallback: black image (rare edge case)
            image = Image.fromarray(np.zeros((448, 448, 3), dtype=np.uint8))

        encoded = processor(
            text=report,
            images=image,
            padding="max_length",
            max_length=64,
            truncation=True,
            return_tensors="pt",
        )

        return {
            "input_ids":      encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
            "pixel_values":   encoded["pixel_values"].squeeze(0),
            "report":         report,
            "img_path":       str(scan_dir),
        }

In [ ]:
class CheXpertDataset(torch.utils.data.Dataset):
    """CheXpert dataset — identical logic to v1 but uses AutoProcessor."""

    def __init__(self, df):
        df = df.reset_index(drop=True)
        self.reports   = df.apply(generate_chexpert_report, axis=1).tolist()
        self.img_paths = df["Path"].tolist()

    def __len__(self):
        return len(self.reports)

    def __getitem__(self, idx):
        report   = self.reports[idx]
        img_path = self.img_paths[idx]

        image = Image.open(img_path).convert("RGB")

        encoded = processor(
            text=report,
            images=image,
            padding="max_length",
            max_length=64,
            truncation=True,
            return_tensors="pt",
        )

        return {
            "input_ids":      encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
            "pixel_values":   encoded["pixel_values"].squeeze(0),
            "report":         report,
            "img_path":       img_path,
        }

In [ ]:
# ─── CheXpert ────────────────────────────────────────────────────────────
chexpert_train = CheXpertDataset(train_full)
chexpert_val   = CheXpertDataset(val_full)

# ─── CQ500 ───────────────────────────────────────────────────────────────
# CQ500 has no official train/val split — we do a manual 80/20 split.
cq500_full = CQ500Dataset(CQ500_ROOT, CQ500_READS_CSV)

cq500_n_val   = max(1, int(0.2 * len(cq500_full)))
cq500_n_train = len(cq500_full) - cq500_n_val
cq500_train, cq500_val = torch.utils.data.random_split(
    cq500_full,
    [cq500_n_train, cq500_n_val],
    generator=torch.Generator().manual_seed(42)
)

# ─── Combined datasets ───────────────────────────────────────────────────
# ConcatDataset stacks both; the training loop treats them identically
# because every item has the same dict keys.
train_dataset = ConcatDataset([chexpert_train, cq500_train])
val_dataset   = ConcatDataset([chexpert_val,   cq500_val])

# Batch size 16 works for 448x448 on T4; increase to 32 on A100
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

print(f"Combined train: {len(train_dataset):,} samples in {len(train_loader)} batches")
print(f"Combined val:   {len(val_dataset):,} samples in {len(val_loader)} batches")

In [ ]:
def siglip_loss(image_embeds, text_embeds, logit_scale, logit_bias):
    """Sigmoid contrastive loss (SigLIP). Symmetric across both modalities."""
    logits = torch.matmul(image_embeds, text_embeds.T) * logit_scale.exp() + logit_bias
    n = logits.shape[0]
    labels = 2.0 * torch.eye(n, device=logits.device) - 1.0
    loss = -F.logsigmoid(labels * logits).sum() / n
    return loss

In [ ]:
def recall_at_k(image_embeds, text_embeds, k_values=[1, 5]):
    similarity = torch.matmul(image_embeds, text_embeds.T)
    results = {}
    for k in k_values:
        top_k = similarity.topk(k, dim=1).indices
        correct = torch.tensor([i in top_k[i] for i in range(len(image_embeds))], dtype=torch.float)
        results[f"image_to_text_recall@{k}"] = correct.mean().item()
    for k in k_values:
        top_k = similarity.T.topk(k, dim=1).indices
        correct = torch.tensor([i in top_k[i] for i in range(len(text_embeds))], dtype=torch.float)
        results[f"text_to_image_recall@{k}"] = correct.mean().item()
    return results

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

SAVE_PATH = "/content/drive/MyDrive/models/medsiglip_lora_chexpert_cq500/"
os.makedirs(SAVE_PATH, exist_ok=True)

### Training Loop
**Fix 1:** `torch.amp.autocast("cuda")` replaces the deprecated `torch.cuda.amp.autocast()`.  
**Fix 2:** Only LoRA parameters + logit_scale/bias are in the optimizer — the  
frozen text encoder and base vision weights receive no gradient updates.

In [ ]:
# ─── Optimizer ───────────────────────────────────────────────────────────
# Only trainable parameters (LoRA weights + logit_scale/bias)
trainable_params = [p for p in model.parameters() if p.requires_grad]
print(f"Optimising {sum(p.numel() for p in trainable_params):,} parameters")

optimizer = torch.optim.AdamW(trainable_params, lr=2e-4, weight_decay=0.01)

# Cosine LR schedule — helps LoRA converge cleanly
total_steps = len(train_loader) * 4   # 4 epochs
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps)

scaler = torch.cuda.amp.GradScaler()

# ─── Hyperparameters ─────────────────────────────────────────────────────
max_epoch_number = 4
test_freq  = 500   # validate every N iterations
ckpt_freq  = 500

# ─── Training ────────────────────────────────────────────────────────────
iteration = 0
for epoch in range(max_epoch_number):
    model.train()
    batch_losses = []

    for batch in train_loader:
        pixel_values   = batch["pixel_values"].to(device)
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        optimizer.zero_grad()

        # ── FIX 1: torch.amp.autocast("cuda") ──────────────────────────
        with torch.amp.autocast("cuda"):
            outputs = model(
                pixel_values=pixel_values,
                input_ids=input_ids,
                attention_mask=attention_mask,
            )
            image_embeds = outputs.image_embeds
            text_embeds  = outputs.text_embeds

            if torch.isnan(image_embeds).any():
                print("NaN in image_embeds"); break
            if torch.isnan(text_embeds).any():
                print("NaN in text_embeds");  break

            loss = siglip_loss(image_embeds, text_embeds, model.logit_scale, model.logit_bias)

            if torch.isnan(loss):
                print("NaN in loss"); break

        batch_loss_value = loss.item()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        batch_losses.append(batch_loss_value)

        # ── Validation ───────────────────────────────────────────────────
        if iteration % test_freq == 0:
            model.eval()
            val_losses, all_img_e, all_txt_e = [], [], []

            with torch.no_grad():
                for batch_val in val_loader:
                    vp  = batch_val["pixel_values"].to(device)
                    vid = batch_val["input_ids"].to(device)
                    vam = batch_val["attention_mask"].to(device)

                    with torch.amp.autocast("cuda"):
                        vout = model(pixel_values=vp, input_ids=vid, attention_mask=vam)
                        vloss = siglip_loss(vout.image_embeds, vout.text_embeds,
                                            model.logit_scale, model.logit_bias)

                    val_losses.append(vloss.item())
                    all_img_e.append(vout.image_embeds.cpu().float())
                    all_txt_e.append(vout.text_embeds.cpu().float())

            all_img_e = torch.cat(all_img_e, dim=0)
            all_txt_e = torch.cat(all_txt_e, dim=0)

            avg_val_loss = float(np.mean(val_losses))
            recall = {k: round(v, 5) for k, v in recall_at_k(all_img_e, all_txt_e).items()}
            print(f"epoch:{epoch+1:2d} iter:{iteration:4d} val_loss:{avg_val_loss:.3f}  recall@k:{recall}")
            model.train()

        # ── Checkpoint ───────────────────────────────────────────────────
        if iteration % ckpt_freq == 0:
            ckpt_path = os.path.join(SAVE_PATH, f"checkpoint_iter_{iteration}.pt")
            torch.save({
                "epoch":      epoch,
                "iteration":  iteration,
                "lora_state": model.vision_model.state_dict(),  # only LoRA weights needed
                "logit_scale": model.logit_scale.data,
                "logit_bias":  model.logit_bias.data,
                "optimizer":  optimizer.state_dict(),
                "scaler":     scaler.state_dict(),
            }, ckpt_path)
            print(f"Checkpoint saved: {ckpt_path}")

        iteration += 1

    train_loss = float(np.mean(batch_losses))
    print(f"epoch:{epoch+1:2d} iter:{iteration:4d} train_loss:{train_loss:.3f}\n")

# ─── Final checkpoint ────────────────────────────────────────────────────
final_path = os.path.join(SAVE_PATH, "final_lora_checkpoint.pt")
torch.save({
    "epoch":      epoch,
    "lora_state": model.vision_model.state_dict(),
    "logit_scale": model.logit_scale.data,
    "logit_bias":  model.logit_bias.data,
    "optimizer":  optimizer.state_dict(),
    "scaler":     scaler.state_dict(),
    "train_loss": train_loss,
}, final_path)
print(f"Final checkpoint saved: {final_path}")

### Final Validation Evaluation (before HF upload)

In [ ]:
model.eval()
final_val_losses, final_img_e, final_txt_e = [], [], []

with torch.no_grad():
    for batch_val in val_loader:
        vp  = batch_val["pixel_values"].to(device)
        vid = batch_val["input_ids"].to(device)
        vam = batch_val["attention_mask"].to(device)

        with torch.amp.autocast("cuda"):
            vout  = model(pixel_values=vp, input_ids=vid, attention_mask=vam)
            vloss = siglip_loss(vout.image_embeds, vout.text_embeds,
                                model.logit_scale, model.logit_bias)

        final_val_losses.append(vloss.item())
        final_img_e.append(vout.image_embeds.cpu().float())
        final_txt_e.append(vout.text_embeds.cpu().float())

final_img_e = torch.cat(final_img_e, dim=0)
final_txt_e = torch.cat(final_txt_e, dim=0)

final_val_loss = float(np.mean(final_val_losses))
final_recall   = {k: round(v, 5) for k, v in recall_at_k(final_img_e, final_txt_e).items()}

print(f"Final val loss : {final_val_loss:.4f}")
print(f"Final recall@k : {final_recall}")

# Save eval results alongside the checkpoint
eval_results = {"val_loss": final_val_loss, **final_recall}
with open(os.path.join(SAVE_PATH, "eval_results.json"), "w") as f:
    json.dump(eval_results, f, indent=2)
print("Eval results saved.")

### Upload Fine-tuned Model to HuggingFace Hub

We upload **only the LoRA adapter weights** (a few MB), not the 800M base model.  
Anyone loading the model needs to:
1. Download the base `google/medsiglip-448` weights
2. Load these adapter weights on top via PEFT

This is the standard workflow for sharing LoRA fine-tunes.

In [ ]:
# ─── Configuration ───────────────────────────────────────────────────────
HF_REPO_ID = "your-username/medsiglip-chexpert-cq500-lora"  # ← change this
HF_TOKEN   = userdata.get("HF_TOKEN")

# ─── Create repo (if it doesn't exist) ──────────────────────────────────
api = HfApi()
api.create_repo(
    repo_id=HF_REPO_ID,
    token=HF_TOKEN,
    exist_ok=True,
    private=False,  # set True if you want a private repo
)
print(f"Repo ready: https://huggingface.co/{HF_REPO_ID}")

# ─── Save adapter + metadata locally ────────────────────────────────────
upload_dir = os.path.join(SAVE_PATH, "hf_upload")
os.makedirs(upload_dir, exist_ok=True)

# 1. Save the PEFT adapter (LoRA weights only — small!)
model.vision_model.save_pretrained(upload_dir)

# 2. Save logit_scale and logit_bias as a small extras file
torch.save(
    {"logit_scale": model.logit_scale.data, "logit_bias": model.logit_bias.data},
    os.path.join(upload_dir, "logit_extras.pt")
)

# 3. Save eval results
import shutil
shutil.copy(os.path.join(SAVE_PATH, "eval_results.json"), upload_dir)

# 4. Write a minimal README / model card
readme = f"""---
base_model: google/medsiglip-448
library_name: peft
license: health-ai-developer-foundations
tags:
  - medical
  - radiology
  - siglip
  - lora
  - chest-x-ray
  - head-ct
---

# MedSigLIP LoRA — CheXpert + CQ500

LoRA fine-tune of [google/medsiglip-448](https://huggingface.co/google/medsiglip-448)
on the CheXpert chest X-ray dataset and CQ500 head CT dataset.

## Training details
- Base model: google/medsiglip-448 (400M vision + 400M text encoder)
- Fine-tuning strategy: LoRA (r=16, alpha=32) on vision encoder attention layers only
- Text encoder: frozen throughout training
- Datasets: CheXpert (chest X-ray, ~223K train) + CQ500 (head CT, ~391 train)
- Loss: SigLIP sigmoid contrastive loss
- Epochs: 4

## Evaluation results
{json.dumps(eval_results, indent=2)}

## Usage
```python
from transformers import AutoProcessor, AutoModel
from peft import PeftModel

base = AutoModel.from_pretrained("google/medsiglip-448")
base.vision_model = PeftModel.from_pretrained(base.vision_model, "{HF_REPO_ID}")
processor = AutoProcessor.from_pretrained("google/medsiglip-448")
```
"""

with open(os.path.join(upload_dir, "README.md"), "w") as f:
    f.write(readme)

# ─── Upload ──────────────────────────────────────────────────────────────
api.upload_folder(
    folder_path=upload_dir,
    repo_id=HF_REPO_ID,
    token=HF_TOKEN,
    commit_message=f"LoRA adapter: CheXpert+CQ500, val_loss={final_val_loss:.4f}",
)
print(f"\nModel uploaded: https://huggingface.co/{HF_REPO_ID}")

### Inference — Image-to-Image & Text-to-Image Retrieval

Both retrieval directions work out of the box:
- **Image-to-image**: encode query image, cosine-nearest-neighbour over the database
- **Text-to-image**: encode a free-text query string; text encoder is frozen so any 
  query phrasing that MedSigLIP understood before fine-tuning still works

In [ ]:
@torch.no_grad()
def embed_images(img_sources, batch_size=32):
    """
    Embed a list of image paths (str) or PIL Images using the vision encoder.
    Returns (N, D) L2-normalised float32 tensor.
    """
    model.eval()
    all_embeds = []

    for start in range(0, len(img_sources), batch_size):
        batch_src = img_sources[start : start + batch_size]
        images = []
        for src in batch_src:
            img = Image.open(src).convert("RGB") if isinstance(src, (str, Path)) else src
            images.append(img)

        inputs = processor(images=images, return_tensors="pt").to(device)

        with torch.amp.autocast("cuda"):
            vis_out = model.vision_model(**inputs)
            embeds  = model.visual_projection(vis_out.pooler_output)
            embeds  = F.normalize(embeds, p=2, dim=-1)

        all_embeds.append(embeds.cpu().float())

    return torch.cat(all_embeds, dim=0)


@torch.no_grad()
def embed_text(text_queries, batch_size=64):
    """
    Embed a list of text strings using the (frozen) text encoder.
    Returns (N, D) L2-normalised float32 tensor.
    """
    model.eval()
    all_embeds = []

    for start in range(0, len(text_queries), batch_size):
        batch_txt = text_queries[start : start + batch_size]
        inputs = processor(
            text=batch_txt, padding="max_length", max_length=64,
            truncation=True, return_tensors="pt"
        ).to(device)

        with torch.amp.autocast("cuda"):
            txt_out = model.text_model(
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"]
            )
            embeds = model.text_projection(txt_out.pooler_output)
            embeds = F.normalize(embeds, p=2, dim=-1)

        all_embeds.append(embeds.cpu().float())

    return torch.cat(all_embeds, dim=0)


def retrieve(query_embed, db_embeds, db_ids, top_k=5):
    """Return top_k (id, score) pairs from the database given a single query embed."""
    sims  = (query_embed @ db_embeds.T).squeeze(0)
    top   = sims.topk(top_k)
    return [(db_ids[i], sims[i].item()) for i in top.indices.tolist()]


# ─── Example: image-to-image retrieval ──────────────────────────────────
# db_paths  = val_full["Path"].tolist()
# db_embeds = embed_images(db_paths)
# torch.save(db_embeds, os.path.join(SAVE_PATH, "val_image_embeddings.pt"))
# results   = retrieve(embed_images([db_paths[0]]), db_embeds, db_paths, top_k=5)
# for path, score in results:
#     print(f"{score:.4f}  {path}")

# ─── Example: text-to-image retrieval (preserved by frozen text encoder) ─
# query_embed = embed_text(["frontal chest radiograph showing pleural effusion"])
# results = retrieve(query_embed, db_embeds, db_paths, top_k=5)
# for path, score in results:
#     print(f"{score:.4f}  {path}")